# Yabbi (my.yabbi.me) — demo

Notebook для работы с функциями из `yabbi_automate.py`:

1. `get_campaign_dict()` — справочник кампаний
2. `get_campaigns_daily_stat(date_from, date_to)` — статистика по кампаниям по дням
3. `get_banners_daily_stat(date_from, date_to)` — статистика по баннерам по дням (+ campaign_id)

Колонки и обогащение таблиц — по стандарту avito (snake_case, `impressions`/`clicks`/`costs_nds`,
`video_views_*`, ключи `id_key_*`; сводка — `../info/00_yabbi_source.md` §5.0).

## Перед запуском
1. Установи зависимости (ячейка ниже)
2. Добавь учётные данные в файл `.env` (по образцу `.env.example`):
   `YABBI_LOGIN`, `YABBI_PASSWORD`, `YABBI_GLOBAL_START_DATE`

## 1. Установка зависимостей (один раз)

In [ ]:
%pip install -q -r requirements.txt

## 2. Импорты и загрузка учётных данных

In [ ]:
import logging
import os
from pathlib import Path


def load_env(path):
    """Мини-загрузчик .env (библиотека dotenv не читает; utf-8-sig терпит BOM)."""
    p = Path(path)
    if not p.exists():
        return False
    for line in p.read_text(encoding="utf-8-sig").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        os.environ.setdefault(k.strip(), v.strip())
    return True


# .env ищем рядом с ноутбуком, затем в корне репо
load_env(".env") or load_env("../.env")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

assert os.environ.get("YABBI_LOGIN"), "YABBI_LOGIN отсутствует в .env"
assert os.environ.get("YABBI_PASSWORD"), "YABBI_PASSWORD отсутствует в .env"
assert os.environ.get("YABBI_GLOBAL_START_DATE"), "YABBI_GLOBAL_START_DATE отсутствует в .env"

print("Учётные данные загружены")

In [ ]:
import sys

sys.path.insert(0, ".")  # notebook лежит рядом с yabbi_automate.py
from yabbi_automate import (
    get_campaign_dict,
    get_campaigns_daily_stat,
    get_banners_daily_stat,
)

## 3. Параметры периода

> **Важно про скорость:** `/report-ajax` тратит ~5–7 секунд на КАЖДУЮ кампанию (id уходят
> батчами по 5), поэтому один день на ~40 кампаниях — это ≈4–5 минут на функцию статистики.
> По умолчанию период = 1 день (вчера); расширяй осознанно.

In [ ]:
from datetime import date, timedelta

DATE_TO = (date.today() - timedelta(days=1)).isoformat()
DATE_FROM = DATE_TO  # 1 день по умолчанию — см. примечание про скорость выше

# Переопредели вручную если нужно:
# DATE_FROM = "2026-07-01"
# DATE_TO   = "2026-07-02"

print(f"Период: {DATE_FROM} → {DATE_TO}")

## 4. Справочник кампаний

In [ ]:
get_campaign_dict_result = get_campaign_dict()
print(f"Строк: {len(get_campaign_dict_result)}")
get_campaign_dict_result.head(10)

In [ ]:
# Быстрый анализ
if not get_campaign_dict_result.empty:
    print(f"Всего записей: {len(get_campaign_dict_result)}")
    # При необходимости добавь анализ по конкретным колонкам:
    # get_campaign_dict_result["status"].value_counts()

## 5. Статистика по кампаниям по дням

Каждый день забирается отдельным запросом (окно `[D, D+1]` МСК + фильтр по дню),
id кампаний — батчами по 5. Имён кампаний в таблице нет — при необходимости
join со справочником по `campaign_id`.

> ⚠ `costs_nds` хранит расход источника КАК ЕСТЬ (Yabbi отдаёт без НДС) —
> см. `../info/00_yabbi_source.md` §5.0.

In [ ]:
get_campaigns_daily_stat_result = get_campaigns_daily_stat(DATE_FROM, DATE_TO)
print(f"Строк: {len(get_campaigns_daily_stat_result)}")
get_campaigns_daily_stat_result.head(10)

In [ ]:
# Сводка — суммы метрик по дням
if not get_campaigns_daily_stat_result.empty:
    summary = get_campaigns_daily_stat_result.groupby("date")[
        ["impressions", "load", "clicks",
         "costs_nds", "costs_without_nds", "costs_nds_ak", "costs_without_nds_ak"]
    ].sum()
    display(summary)

## 6. Статистика по баннерам по дням

Идентификатор баннера — `url` (своего id у баннера нет); строки агрегированы суммой
по `(date, url)`. `campaign_id` подтянут через `campaigns-banners-daily`;
у баннеров без найденной кампании `campaign_id`/`id_key_camp`/`id_key_ad` = None.

In [ ]:
get_banners_daily_stat_result = get_banners_daily_stat(DATE_FROM, DATE_TO)
print(f"Строк: {len(get_banners_daily_stat_result)}")
get_banners_daily_stat_result.head(10)

In [ ]:
# Сводка — суммы метрик по дням
if not get_banners_daily_stat_result.empty:
    summary = get_banners_daily_stat_result.groupby("date")[
        ["impressions", "clicks", "video_views_100"]
    ].sum()
    display(summary)

## Сохранение в CSV (опционально)

In [ ]:
from datetime import date
from pathlib import Path

today = date.today().isoformat()

# имена файлов содержат дату/период — старые выгрузки сами не перезаписываются, удаляем
for fn_name in ["get_campaign_dict", "get_campaigns_daily_stat", "get_banners_daily_stat"]:
    for old in Path(".").glob(f"{fn_name}_*.csv"):
        old.unlink()

get_campaign_dict_result.to_csv(
    f"get_campaign_dict_{today}.csv", index=False, encoding="cp1251", errors="replace")
get_campaigns_daily_stat_result.to_csv(
    f"get_campaigns_daily_stat_{DATE_FROM}_{DATE_TO}.csv", index=False, encoding="cp1251", errors="replace")
get_banners_daily_stat_result.to_csv(
    f"get_banners_daily_stat_{DATE_FROM}_{DATE_TO}.csv", index=False, encoding="cp1251", errors="replace")

print("Файлы сохранены")